# Modeling Notebook

Train multiple classifiers and compare results.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import joblib

base_dir = Path.cwd().parent
features = pd.read_csv(base_dir / 'data' / 'processed' / 'user_features.csv')
target = 'activity_level'

drop_cols = {'user_id', 'activity_threshold_low', 'activity_threshold_high'}
x = features.drop(columns=[target] + [col for col in drop_cols if col in features.columns])
y = features[target]

numeric_cols = x.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in x.columns if col not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ]
)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

models = {
    'logistic_regression': LogisticRegression(max_iter=200, class_weight='balanced'),
    'decision_tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
}

results = []
best_name = None
best_f1 = -1
best_model = None

for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(x_train, y_train)
    preds = pipeline.predict(x_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision_macro': precision_score(y_test, preds, average='macro', zero_division=0),
        'recall_macro': recall_score(y_test, preds, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, preds, average='macro', zero_division=0),
    })
    if results[-1]['f1_macro'] > best_f1:
        best_f1 = results[-1]['f1_macro']
        best_name = name
        best_model = pipeline

results_df = pd.DataFrame(results)
display(results_df)

if best_model is not None:
    Path('models').mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, 'models/best_model.joblib')
    print('Best model:', best_name, 'F1:', best_f1)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/user_features.csv'